In [44]:
#importing libraries
import numpy as np
import pandas as pd


In [45]:
#reading dataset
df1=pd.read_csv('matches.csv')
df2=pd.read_csv('deliveries.csv')


In [46]:
#shape of dataset
df1.shape


(1095, 20)

In [47]:
df2.shape

(260920, 17)

In [48]:
#first five columns of the dataset
df1.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [49]:
df2.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [50]:
# Drop unnecessary columns
columns_to_drop = [
    'season', 'city', 'date', 'match_type', 'player_of_match',
    'team1', 'team2', 'toss_winner', 'toss_decision',
    'result', 'result_margin', 'target_runs', 'target_overs',
    'super_over', 'method', 'umpire1', 'umpire2'
]
df1.drop(columns=columns_to_drop, inplace=True)

In [51]:
# Standardize team names using a mapping dictionary
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}
df1['winner'] = df1['winner'].replace(team_mapping)

In [52]:
# Map the original venue column to a standardized version
venue_mapping = {
    'Punjab Cricket Association Stadium, Mohali': 'PCA Stadium',
    'M Chinnaswamy Stadium': 'Chinnaswamy Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    'Eden Gardens': 'Eden Gardens (Kolkata)'
}
df1['venue_canonical'] = df1['venue'].replace(venue_mapping)
df1.drop(columns=['venue'], inplace=True)

In [53]:
# Retain only the required columns
df1 = df1[['id', 'winner', 'venue_canonical']]

# Display the final cleaned dataframe
print(df1.head())

       id                       winner         venue_canonical
0  335982        Kolkata Knight Riders     Chinnaswamy Stadium
1  335983          Chennai Super Kings             PCA Stadium
2  335984               Delhi Capitals    Arun Jaitley Stadium
3  335985  Royal Challengers Bangalore        Wankhede Stadium
4  335986        Kolkata Knight Riders  Eden Gardens (Kolkata)


In [54]:
# Save the cleaned dataset
df1.to_csv('cleaned_matches.csv', index=False)

In [55]:
#Subset data to keep essential columns
essential_columns = [
    'match_id', 'inning', 'batting_team', 'bowling_team',
    'over', 'ball', 'total_runs', 'is_wicket'
]
df2 = df2[essential_columns]

In [56]:
#Standardize team names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}
df2['batting_team'] = df2['batting_team'].replace(team_mapping)
df2['bowling_team'] = df2['bowling_team'].replace(team_mapping)

<ipython-input-56-939a9b2ce9d4>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['batting_team'] = df2['batting_team'].replace(team_mapping)


In [57]:
#Compute aggregated features
# 1.Cumulative runs and wickets
df2['cum_runs'] = df2.groupby(['match_id', 'inning'])['total_runs'].cumsum()
df2['cum_wickets'] = df2.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

# 2. Overs completed
df2['overs_completed'] = df2['over'] + (df2['ball'] / 6)

# 3.Current run rate (avoid division by zero)
df2['current_run_rate'] = df2['cum_runs'] / df2['overs_completed']
df2['current_run_rate'].fillna(0, inplace=True)

# 4.Display the final dataset
print(df2.head())

   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball  total_runs  is_wicket  cum_runs  cum_wickets  overs_completed  \
0     1           1          0         1            0         0.166667   
1     2           0          0         1            0         0.333333   
2     3           1          0         2            0         0.500000   
3     4           0          0         2            0         0.666667   
4     5           0          0         2            0         0.833333   

   current_run_rate  
0               6.0  
1               3.0  
2 

<ipython-input-57-f6b3cc8a31e9>:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df2['current_run_rate'].fillna(0, inplace=True)


In [58]:
#Save the processed data
df2.to_csv('processed_deliveries.csv', index=False)

In [59]:
#Reading the datsset
deliveries = pd.read_csv('processed_deliveries.csv')
matches = pd.read_csv('cleaned_matches.csv')

In [60]:
#Merge on 'match_id' and 'id' using a left join
merged_df = deliveries.merge(matches, left_on='match_id', right_on='id', how='left')



In [61]:
print(merged_df.head())

   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball  total_runs  is_wicket  cum_runs  cum_wickets  overs_completed  \
0     1           1          0         1            0         0.166667   
1     2           0          0         1            0         0.333333   
2     3           1          0         2            0         0.500000   
3     4           0          0         2            0         0.666667   
4     5           0          0         2            0         0.833333   

   current_run_rate      id                 winner      venue_canoni

In [62]:
#Save the merged dataset
merged_df.to_csv('merged_dataset.csv', index=False)

In [63]:
# Load the merged dataset
file_path = 'merged_dataset.csv'
df = pd.read_csv(file_path)


In [64]:
#  Create binary target variable
df['win'] = (df['batting_team'] == df['winner']).astype(int)

In [65]:
# Filter for second innings only
df = df[df['inning'] == 2]

In [66]:
#  Compute target and required run rate
df['target'] = df.groupby('match_id')['cum_runs'].transform('max') + 1
df['required_run_rate'] = np.where(
    (20 - df['overs_completed']) > 0,
    (df['target'] - df['cum_runs']) / (20 - df['overs_completed']),
    0
)

In [67]:
#  Select final features
final_df = df[[
    'match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
    'required_run_rate', 'target', 'batting_team', 'bowling_team',
    'venue_canonical', 'win'
]]

In [68]:
#Display the final processed data
print(final_df.head())


     match_id  inning  cum_runs  cum_wickets  current_run_rate  \
124    335982       2         1            0               6.0   
125    335982       2         2            0               6.0   
126    335982       2         2            0               4.0   
127    335982       2         3            0               4.5   
128    335982       2         4            0               4.8   

     required_run_rate  target                 batting_team  \
124           4.134454      83  Royal Challengers Bangalore   
125           4.118644      83  Royal Challengers Bangalore   
126           4.153846      83  Royal Challengers Bangalore   
127           4.137931      83  Royal Challengers Bangalore   
128           4.121739      83  Royal Challengers Bangalore   

              bowling_team      venue_canonical  win  
124  Kolkata Knight Riders  Chinnaswamy Stadium    0  
125  Kolkata Knight Riders  Chinnaswamy Stadium    0  
126  Kolkata Knight Riders  Chinnaswamy Stadium    0  
127 

In [69]:
#Save the final dataset
final_df.to_csv('final_dataset.csv', index=False)